**MQTT Topics:**

| Topic | Direction | Payload |
|-------|-----------|---------|
| `team08/hospital/control` | Frontend sends | `{"command": "select_drone", "drone": "drone-01"}` |
| `team08/hospital/control` | Frontend sends | `{"command": "medicine_loaded", "drone": "drone-01"}` |
| `team08/hospital/control` | Frontend sends | `{"command": "return", "drone": "drone-01"}` |
| `team08/hospital/control` | Frontend sends | `{"command": "back"}` |
| `team08/hospital/display` | Backend publishes | `{"status": "selected_drone", "drone": "drone-01", "message": "..."}` |
| `team08/hospital/display` | Backend publishes | `{"status": "medicine_loaded", "drone": "drone-01", "message": "..."}` |
| `team08/hospital/display` | Backend publishes | `{"status": "returning", "drone": "drone-01", "message": "..."}` |
| `team08/hospital/display` | Backend publishes | `{"status": "initiated", "message": "..."}` |

In [5]:
import json
from threading import Thread

import paho.mqtt.client as mqtt
from stmpy import Driver, Machine

In [ ]:
broker, port = "mqtt20.iik.ntnu.no", 1883

MQTT_TOPIC_CONTROL = "team08/hospital/control"
MQTT_TOPIC_DISPLAY = "team08/hospital/display"
MQTT_TOPIC_USER    = "team08/hospital/user"
 
 
class HospitalFrontend:
 
    def display_drone(self, drone):
        message = "Drone {} selected.".format(drone)
        print(message)
        payload = json.dumps({"status": "selected_drone", "drone": drone, "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)
 
    def display_overview(self):
        message = "Back to overview. Showing all drones."
        print(message)
        payload = json.dumps({"status": "overview_all_drones", "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)
 
    def transit_to_drone_return(self, drone):
        message = "Drone {} is returning.".format(drone)
        print(message)
        # FIX: publish to DISPLAY so the hospital UI sees the status change
        payload = json.dumps({"status": "returning", "drone": drone, "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)
 
    def transmit_medicine_loaded(self, drone):
        message = "Medicine loaded on drone {}.".format(drone)
        print(message)
        # FIX: publish to DISPLAY so the hospital UI sees the status change
        payload = json.dumps({"status": "loaded", "drone": drone, "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)
 
    def handle_request(self, drone, medicine):
        message = "Drone {} requested with medicine: {}.".format(drone, medicine)
        print(message)
        payload = json.dumps({
            "status":   "requested",
            "drone":    drone,
            "medicine": medicine,
            "message":  message,
        })
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)
 
    def handle_returned(self, drone):
        """NEW: called when a drone reports it has returned — resets to idle."""
        message = "Drone {} has returned. Now idle.".format(drone)
        print(message)
        payload = json.dumps({"status": "returned", "drone": drone, "message": message})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)
 
    def on_init(self):
        print("Hospital frontend initialized. Showing overview of all drones.")
        payload = json.dumps({"status": "initiated", "message": "System is initiated."})
        self.mqtt_client.publish(MQTT_TOPIC_DISPLAY, payload)


In [7]:
drone_frontend = HospitalFrontend()
 
# Init
t0 = {
    "source": "initial",
    "target": "overview_all_drones",
    "effect": "on_init",
}
 
# User selects a drone
t1 = {
    "trigger": "select_drone",
    "source":  "overview_all_drones",
    "target":  "selected_drone",
    "effect":  "display_drone(*)",
}
 
# Back to overview
t2 = {
    "trigger": "back",
    "source":  "selected_drone",
    "target":  "overview_all_drones",
    "effect":  "display_overview",
}
 
# Drone returning
t3 = {
    "trigger": "return",
    "source":  "selected_drone",
    "target":  "selected_drone",
    "effect":  "transit_to_drone_return(*)",
}
 
# Medicine loaded
t4 = {
    "trigger": "medicine_loaded",
    "source":  "selected_drone",
    "target":  "selected_drone",
    "effect":  "transmit_medicine_loaded(*)",
}
 
# NEW: user requests a drone (can happen from overview or selected state)
# args: drone, medicine
t5 = {
    "trigger": "request",
    "source":  "overview_all_drones",
    "target":  "overview_all_drones",
    "effect":  "handle_request(*)",
}
 
t6 = {
    "trigger": "request",
    "source":  "selected_drone",
    "target":  "selected_drone",
    "effect":  "handle_request(*)",
}
 
# NEW: drone has physically returned → publish "returned" so UI goes back to idle
t7 = {
    "trigger": "returned",
    "source":  "selected_drone",
    "target":  "selected_drone",
    "effect":  "handle_returned(*)",
}
 
t8 = {
    "trigger": "returned",
    "source":  "overview_all_drones",
    "target":  "overview_all_drones",
    "effect":  "handle_returned(*)",
}
 
drone_machine = Machine(
    name="hospital_frontend",
    transitions=[t0, t1, t2, t3, t4, t5, t6, t7, t8],
    obj=drone_frontend,
)
drone_frontend.stm = drone_machine
 
 
class HospitalMQTTClient:
    
    def __init__(self):
        self.client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
        self.client.on_connect = self.on_connect
        self.client.on_message = self.on_message

    def on_connect(self, client, userdata, flags, reason_code, properties):
        print("on_connect(): {}".format(reason_code))
        # Subscribe to both — request may come from either topic
        self.client.subscribe(MQTT_TOPIC_CONTROL)
        self.client.subscribe(MQTT_TOPIC_USER)
        print("Subscribed to: {} and {}".format(MQTT_TOPIC_CONTROL, MQTT_TOPIC_USER))

    def on_message(self, client, userdata, msg):
        print("on_message(): topic: {} payload: {}".format(
            msg.topic, msg.payload.decode("utf-8")))

        try:
            payload = json.loads(msg.payload.decode("utf-8"))
        except json.JSONDecodeError:
            print("Invalid JSON payload, ignoring.")
            return

        command  = payload.get("command", "")
        drone    = payload.get("drone", "Unknown")
        medicine = payload.get("medicine", "")
        status   = payload.get("status", "")

        # ── request: handle regardless of which topic it arrives on ──────────
        # This makes it work whether the user UI publishes on /user or /control
        if command == "request" or status == "requested":
            print("Request: drone={} medicine={}".format(drone, medicine))
            self.stm_driver.send("request", "hospital_frontend", args=[drone, medicine])

        # ── returned: handle regardless of topic ─────────────────────────────
        elif command == "returned" or status == "returned":
            print("Returned: drone={}".format(drone))
            self.stm_driver.send("returned", "hospital_frontend", args=[drone])

        # ── hospital UI button commands ───────────────────────────────────────
        elif command == "select_drone":
            print("Select drone: {}".format(drone))
            self.stm_driver.send("select_drone", "hospital_frontend", args=[drone])

        elif command == "medicine_loaded":
            print("Medicine loaded: drone={}".format(drone))
            self.stm_driver.send("medicine_loaded", "hospital_frontend", args=[drone])

        elif command == "return":
            print("Return: drone={}".format(drone))
            self.stm_driver.send("return", "hospital_frontend", args=[drone])

        elif command == "back":
            print("Back to overview")
            self.stm_driver.send("back", "hospital_frontend")

        else:
            print("Unknown command/status: command='{}' status='{}'".format(command, status))

    def start(self, broker, port):
        print("Connecting to {}:{}".format(broker, port))
        self.client.connect(broker, port)
        try:
            thread = Thread(target=self.client.loop_forever)
            thread.start()
        except KeyboardInterrupt:
            print("Interrupted")
            self.client.disconnect()

In [ ]:
driver = Driver()
driver.add_machine(drone_machine)
 
myclient = HospitalMQTTClient()
drone_frontend.mqtt_client = myclient.client
myclient.stm_driver = driver
 
driver.start(keep_active=True)
myclient.start(broker, port)
drone_frontend.on_init()

Hospital frontend initialized. Showing overview of all drones.Connecting to mqtt20.iik.ntnu.no:1883

Hospital frontend initialized. Showing overview of all drones.


on_connect(): Success
Subscribed to: team08/hospital/control and team08/hospital/user


Machine hospital_frontend is in state selected_drone and received event select_drone, but no transition with this event is declared!


on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-01"}on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-01"}
Select drone: drone-01
Drone drone-01 selected.

Selecting drone: drone-01


Machine hospital_frontend is in state selected_drone and received event select_drone, but no transition with this event is declared!
Machine hospital_frontend is in state selected_drone and received event select_drone, but no transition with this event is declared!


...

on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-01"}on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-01"}
Selecting drone: drone-01

Select drone: drone-01


Machine hospital_frontend is in state selected_drone and received event select_drone, but no transition with this event is declared!
Machine hospital_frontend is in state selected_drone and received event select_drone, but no transition with this event is declared!


on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-03"}on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-03"}
Select drone: drone-03

Selecting drone: drone-03


Machine hospital_frontend is in state selected_drone and received event select_drone, but no transition with this event is declared!
Machine hospital_frontend is in state selected_drone and received event select_drone, but no transition with this event is declared!


on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-01"}on_message(): topic: team08/hospital/control payload: {"command": "select_drone", "drone": "drone-01"}
Select drone: drone-01

Selecting drone: drone-01
